# RESITE-GIS AI Hazard Prediction & Susceptibility Model Training
### Autonomous Multi-Hazard Deep Learning & Real-Time Dynamic Trigger Engine
**Target Study Area:** Chamoli District, Uttarakhand, India (`EPSG:32643` UTM Zone 43N / `EPSG:4326`)
**Architecture Adapted from:** `GVCL/Susceptibility-Mapping-FL-Hetero`

This notebook trains and validates the deep neural network hazard susceptibility model using the pre-processed spatial multi-raster feature tensor stack containing:
1. `elevation` (SRTM DEM 90m reprojected to 30m UTM)
2. `slope` (Metric spatial gradient angle in degrees)
3. `aspect` (Metric spatial gradient direction 0–360°)
4. `plan_curvature` (Overland flow divergence/convergence)
5. `profile_curvature` (Down-slope flow acceleration)
6. `twi` (Topographic Wetness Index)
7. `spi` (Stream Power Index)
8. `dist_to_streams` (Euclidean distance to OSM vector waterways)
9. `dist_to_faults` (Euclidean distance to Himalayan MCT tectonic shear zone)
10. `ndvi` (Sentinel-2 Optical L2A 10m NDVI)
11. `lulc` (ESA WorldCover 10m Land Cover classification)
12. `precip_gpm` (NASA GPM IMERG L3 v07 dynamic rainfall trigger)

In [1]:
# Cell 1: Environment Setup & Library Imports
import os
import sys
import json
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import joblib

# Set reproducible seeds
torch.manual_seed(42)
np.random.seed(42)

# Device configuration (GPU if available, CPU for quantized inference)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__} | Execution Device: {device}")

PyTorch Version: 2.13.0+cpu | Execution Device: cpu


In [2]:
# Cell 2: Spatial Data Verification & Dataset Manifest Inspection
raw_dir = "../raw"
proc_dir = "../processed"

summary_path = os.path.join(proc_dir, "dataset_summary.json")
with open(summary_path, "r") as f:
    summary = json.load(f)

print("=== RESITE-GIS Pre-Processed Dataset Manifest ===")
print(f"Project: {summary['project']}")
print(f"Study Area: {summary['study_area']['district']} | CRS: {summary['study_area']['crs']}")
print(f"Master Grid: {summary['study_area']['grid_dimensions']['width']} x {summary['study_area']['grid_dimensions']['height']} ({summary['study_area']['grid_dimensions']['valid_cells']:,} valid land cells)")
print("\nIngested Raw Datasets:")
for k, v in summary['datasets_ingested'].items():
    print(f"  • {k.upper():18s}: {v}")

=== RESITE-GIS Pre-Processed Dataset Manifest ===
Project: RESITE-GIS Hazard Susceptibility Model Pre-Training
Study Area: Chamoli District, Uttarakhand, India | CRS: EPSG:32643 (UTM Zone 43N)
Master Grid: 3860 x 4586 (14,349,161 valid land cells)

Ingested Raw Datasets:
  • DEM               : srtm_52_06.tif (SRTM 90m v4.1 reprojected to 30m)
  • NDVI              : Chamoli_Sentinel2_NDVI_10m.tif (Sentinel-2 Optical L2A 10m resampled to 30m)
  • LULC              : Chamoli_ESA_WorldCover_LULC_10m.tif (ESA WorldCover 10m resampled to 30m)
  • PRECIPITATION     : gpm_v07_precip_2023_7.tif (NASA GPM IMERG L3 v07 resampled to 30m)
  • WATERWAYS         : india-260824.osm.pbf (OSM Vector waterways rasterized to 30m)
  • FAULTS            : Himalayan Main Central Thrust (MCT) Regional Shear Zone Proxy Model
  • LANDSLIDE_CATALOG : Global_Landslide_Catalog_Export_rows.json (NASA GLC Ground Truth)


In [3]:
# Cell 3: Loading Feature Matrix & Training Tensors
tensor_path = os.path.join(proc_dir, "training_tensors.npz")
data_tensors = np.load(tensor_path)
feature_names = data_tensors["feature_names"].tolist()

X_train = data_tensors["X_train"]
y_train = data_tensors["y_train"]
X_val = data_tensors["X_val"]
y_val = data_tensors["y_val"]
X_test = data_tensors["X_test"]
y_test = data_tensors["y_test"]

print(f"Feature Matrix Dimensions (C = {len(feature_names)}):")
for idx, name in enumerate(feature_names):
    fstat = summary['feature_statistics'][name]
    print(f"  [{idx+1:02d}] {name:18s} | Raw Range: [{fstat['min_raw']:8.2f}, {fstat['max_raw']:8.2f}] | Mean: {fstat['mean_raw']:8.2f}")

print(f"\nPartition Statistics:")
print(f"  Training Partition:   {len(X_train):,} samples (Positives: {int((y_train==1).sum())}, Negatives: {int((y_train==0).sum())})")
print(f"  Validation Partition: {len(X_val):,} samples (Positives: {int((y_val==1).sum())}, Negatives: {int((y_val==0).sum())})")
print(f"  Test Partition:       {len(X_test):,} samples (Positives: {int((y_test==1).sum())}, Negatives: {int((y_test==0).sum())})")

Feature Matrix Dimensions (C = 12):
  [01] elevation          | Raw Range: [  624.02,  6794.04] | Mean:  4238.46
  [02] slope              | Raw Range: [    0.00,    47.28] | Mean:    19.74
  [03] aspect             | Raw Range: [    0.00,   359.98] | Mean:   176.31
  [04] plan_curvature     | Raw Range: [   -0.86,     0.77] | Mean:    -0.00
  [05] profile_curvature  | Raw Range: [   -0.02,     0.02] | Mean:     0.00
  [06] twi                | Raw Range: [    0.00,    11.50] | Mean:     6.08
  [07] spi                | Raw Range: [    0.00,   129.39] | Mean:    35.12
  [08] dist_to_streams    | Raw Range: [    0.00, 14074.73] | Mean:  2958.80
  [09] dist_to_faults     | Raw Range: [    1.60, 84083.95] | Mean: 35301.55
  [10] ndvi               | Raw Range: [   -0.35,     0.89] | Mean:     0.19
  [11] lulc               | Raw Range: [   10.00,   100.00] | Mean:    55.19
  [12] precip_gpm         | Raw Range: [  179.59,   767.38] | Mean:   423.52

Partition Statistics:
  Training Partit

In [4]:
# Cell 4: PyTorch Dataset & DataLoader Construction
class GeospatialDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

BATCH_SIZE = 64
train_dataset = GeospatialDataset(X_train, y_train)
val_dataset = GeospatialDataset(X_val, y_val)
test_dataset = GeospatialDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"PyTorch DataLoaders Initialized: Batch Size = {BATCH_SIZE}, Training Batches = {len(train_loader)}")

PyTorch DataLoaders Initialized: Batch Size = 64, Training Batches = 110


In [5]:
# Cell 5: Model Architecture Definition (Deep Multi-Layer Perceptron / 1D Spatial CNN)
class SusceptibilityNN(nn.Module):
    """
    Deep Neural Network for Hazard Susceptibility Mapping
    Adapted from GVCL/Susceptibility-Mapping-FL-Hetero pipeline architecture.
    """
    def __init__(self, input_dim=12):
        super(SusceptibilityNN, self).__init__()
        self.net = nn.Sequential(
            # Layer 1: C -> 128
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # Layer 2: 128 -> 64
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            # Layer 3: 64 -> 32
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            
            # Output Layer: 32 -> 1
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = SusceptibilityNN(input_dim=len(feature_names)).to(device)
print(model)

SusceptibilityNN(
  (net): Sequential(
    (0): Linear(in_features=12, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Linear(in_features=32, out_features=1, bias=True)
    (12): Sigmoid()
  )
)


In [6]:
# Cell 6: Training Setup & Loss/Optimizer Specification
# NOTE: Model is configured and ready to train. Epochs will execute when initiated.
criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for features, targets in loader:
        features, targets = features.to(device), targets.to(device)
        optimizer.zero_grad()
        preds = model(features)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(features)
    return total_loss / len(loader.dataset)

def validate_one_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for features, targets in loader:
            features, targets = features.to(device), targets.to(device)
            preds = model(features)
            loss = criterion(preds, targets)
            total_loss += loss.item() * len(features)
    return total_loss / len(loader.dataset)

print("Training configuration and loss objectives verified. Ready to execute training loop when desired.")

Training configuration and loss objectives verified. Ready to execute training loop when desired.


In [11]:
# Creating training loop and evaluation metrics will be implemented in subsequent cells.

for epoch in range(1, 101):  # Example: 50 epochs
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss = validate_one_epoch(model, val_loader, criterion)
    scheduler.step(val_loss)
    
    print(f"Epoch [{epoch}/100] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

Epoch [1/100] | Train Loss: 0.0048 | Val Loss: 0.0079
Epoch [2/100] | Train Loss: 0.0036 | Val Loss: 0.0074
Epoch [3/100] | Train Loss: 0.0046 | Val Loss: 0.0083
Epoch [4/100] | Train Loss: 0.0051 | Val Loss: 0.0067
Epoch [5/100] | Train Loss: 0.0038 | Val Loss: 0.0072
Epoch [6/100] | Train Loss: 0.0041 | Val Loss: 0.0061
Epoch [7/100] | Train Loss: 0.0057 | Val Loss: 0.0076
Epoch [8/100] | Train Loss: 0.0043 | Val Loss: 0.0073
Epoch [9/100] | Train Loss: 0.0037 | Val Loss: 0.0067
Epoch [10/100] | Train Loss: 0.0037 | Val Loss: 0.0071
Epoch [11/100] | Train Loss: 0.0039 | Val Loss: 0.0070
Epoch [12/100] | Train Loss: 0.0037 | Val Loss: 0.0072
Epoch [13/100] | Train Loss: 0.0042 | Val Loss: 0.0073
Epoch [14/100] | Train Loss: 0.0040 | Val Loss: 0.0070
Epoch [15/100] | Train Loss: 0.0040 | Val Loss: 0.0061
Epoch [16/100] | Train Loss: 0.0044 | Val Loss: 0.0070
Epoch [17/100] | Train Loss: 0.0037 | Val Loss: 0.0066
Epoch [18/100] | Train Loss: 0.0040 | Val Loss: 0.0064
Epoch [19/100] | Tr

In [7]:
# Cell 7: Performance Evaluation Function (ROC-AUC, Precision, Recall, Confusion Matrix)
def evaluate_model(model, loader):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for features, targets in loader:
            features = features.to(device)
            preds = model(features).cpu().numpy()
            all_preds.extend(preds.flatten())
            all_targets.extend(targets.numpy().flatten())
    
    y_true = np.array(all_targets)
    y_score = np.array(all_preds)
    y_pred = (y_score >= 0.5).astype(int)
    
    auc = roc_auc_score(y_true, y_score)
    f1 = f1_score(y_true, y_pred)
    print(f"Model Validation:\n  ROC-AUC: {auc:.4f} (Target > 0.88)\n  F1-Score: {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=["Stable (0)", "Hazard (1)"]))
    return y_true, y_score

print("Evaluation pipeline ready.")

Evaluation pipeline ready.


In [14]:
# Creating an evaluation loop to assess model performance on the validation set will be implemented in subsequent cells.

eval_y_true, eval_y_score = evaluate_model(model, val_loader)

Model Validation:
  ROC-AUC: 1.0000 (Target > 0.88)
  F1-Score: 0.9973

Classification Report:
              precision    recall  f1-score   support

  Stable (0)       0.99      1.00      1.00       750
  Hazard (1)       1.00      0.99      1.00       749

    accuracy                           1.00      1499
   macro avg       1.00      1.00      1.00      1499
weighted avg       1.00      1.00      1.00      1499



In [15]:
# Cell 8: INT8 Dynamic Quantization Export for Low-Latency FastAPI Deployment
def quantize_model(trained_model):
    trained_model.eval()
    quantized = torch.quantization.quantize_dynamic(
        trained_model.cpu(),
        {nn.Linear},
        dtype=torch.qint8
    )
    print("Quantized Model Generated (INT8 Dynamic Quantization).")
    return quantized

print("Quantization pipeline configured.")

Quantization pipeline configured.
